In [43]:
import sqlite3
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

In [2]:
# Connect to the database and load the dataframe
connection = sqlite3.connect('Data/Walmart.db')
table = pd.read_sql('Select * FROM sales_economic_data', con=connection)
table.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,effective_federal_fund,personal_consum_expend,producer_price_index,personal_saving_rate,advanced_retail_sales,consumer_sentiment,household_dept_srvc_ratio
0,5,2010-01-10 00:00:00,283178.12,0,71.10,2.603,212.226946,6.768,0.11,10056.1,181.9,5.6,302325.0,74.4,14.51494
1,15,2010-01-10 00:00:00,566945.95,0,59.69,2.840,132.756800,8.067,0.11,10056.1,181.9,5.6,302325.0,74.4,14.51494
2,42,2010-01-10 00:00:00,481523.93,0,86.01,3.001,126.234600,9.003,0.11,10056.1,181.9,5.6,302325.0,74.4,14.51494
3,33,2010-01-10 00:00:00,224294.39,0,91.45,3.001,126.234600,9.265,0.11,10056.1,181.9,5.6,302325.0,74.4,14.51494
4,36,2010-01-10 00:00:00,422169.47,0,74.66,2.567,210.440443,8.476,0.11,10056.1,181.9,5.6,302325.0,74.4,14.51494


# Group-Wise Statistical Analysis
Here, we statistically examine the weekly sales across different store groups.

1. **In-Group Normality**: The test is executed using Shapiro-Wilk test (```stats.shapiro()```), which evaluates the null hypothesis that the data was originiated from a normal distribution.
2. **Inter-Group Homogeneity**: The Levene’s (```stats.levene()```) test reveals if the variances of groups are equal.

In [3]:
# Check normality within each store type (p-value threshold = 0.05).
ingroup_norm = True
for store, group in table.groupby(by='Store'):
    stat, p_value = stats.shapiro(group['Weekly_Sales'])
    # print(f'Store {store} Shapiro_Wilk p-value: {p_value:0.4f}')
    if p_value < 0.05:
        ingroup_norm = False

# Check for homogeneity of variance across stores (p-value threshold = 0.05)
stat, p_value = stats.levene(*[group['Weekly_Sales'].values for store, group in table.groupby('Store')])
print(f'Store Levene\'s p-value: {p_value:0.4f}')
if p_value < 0.05:
    intergroup_homo = False
else:
    intergroup_homo = True

# Specify the type of analysis
if all([ingroup_norm, intergroup_homo]):
    print('INFO >> In-group distributions are normal and groups are homogeneous. Continue with <ANOVA>, parameteric test.')
else:
    print('INFO >> Joint normality and homogeneity conditions not met. Continue with <Kruskal-Wallis, Mann-Whitney U>, non-parameteric test.')

Store Levene's p-value: 0.0000
INFO >> Joint normality and homogeneity conditions not met. Continue with <Kruskal-Wallis, Mann-Whitney U>, non-parameteric test.


In [ ]:
# Check if at least one store differs significantly in overal sales. It checks whether the medians are equall.
h_stat, p_value = stats.kruskal(*[group['Weekly_Sales'] for _ , group in table.groupby('Store')])
if p_value < 0.05:
    print(f'INFO >> At least one store has a significant difference in sales with others.')

INFO >> At least one store has a significant difference in sales with others.


# Feature Correlation and Statistical Significance


In [41]:
# Normalize sales per store (mean=0, std=1)
table['Sales_Normalized'] = table.groupby('Store')['Weekly_Sales'].transform(lambda x: (x - x.mean()) / x.std())

# Specify the columns to evaluate
features_to_evaluate = ['Fuel_Price', 'household_dept_srvc_ratio',
                'Temperature', 'Unemployment', 
                'effective_federal_fund', 'personal_consum_expend', 
                'producer_price_index', 'personal_saving_rate', 
                'advanced_retail_sales', 'consumer_sentiment',
                'Holiday_Flag'
                ]

# Run the correlation analysis for each pair of sales and numeric features
results = []
alpha = 0.05
for feature in features_to_evaluate:
    r_stat, p_stat = stats.spearmanr(table['Sales_Normalized'], table[feature])
    results.append({
        'feature': feature,
        'corr': r_stat,
        'p_value': p_stat,
        'is_significant': p_stat < alpha
    })
corr_table = pd.DataFrame(results).sort_values(by='p_value', ascending=True)
print(corr_table)
# Determine what features to remove
non_significant_features = corr_table[corr_table['is_significant'] == False]['feature'].to_list()
print(f'INFO >> Correlated but non-significant numeric features are: {non_significant_features}')
table.drop('Sales_Normalized', axis=1, inplace=True)


                      feature      corr       p_value  is_significant
1   household_dept_srvc_ratio -0.098214  2.877412e-15            True
5      personal_consum_expend  0.095171  2.004013e-14            True
10               Holiday_Flag  0.089388  6.791568e-13            True
8       advanced_retail_sales  0.088979  8.640222e-13            True
7        personal_saving_rate  0.077459  4.916476e-10            True
0                  Fuel_Price  0.044812  3.233423e-04            True
3                Unemployment -0.042627  6.254062e-04            True
9          consumer_sentiment  0.040292  1.225611e-03            True
4      effective_federal_fund -0.022795  6.747838e-02           False
6        producer_price_index  0.010428  4.029592e-01           False
2                 Temperature  0.004391  7.247224e-01           False
INFO >> Correlated but non-significant numeric features are: ['effective_federal_fund', 'producer_price_index', 'Temperature']


# time-Aware Feature Engineering
Here, we derive time-aware features to capture chronological sequences/patterns.
1. **Seasonality**: Sine/Cosine cyclical encoding is used to capture seasonal features without using data points.
2. **Rolling Window Aggregation**: it captures the mean of sales for the previous 4 weeks at each Store.

In [51]:
# Seasonality
# Covert date to datetime type, then get the week number of each data point
table['Date'] = pd.to_datetime(table['Date'])
table = table.sort_values(['Store', 'Date']).reset_index(drop=True)
week_series = table['Date'].dt.isocalendar().week

# Convert the week into a smooth 360-degree cyclical coordinates
table['sin_week'] = np.sin(2 * np.pi * week_series / 52.1429)
table['cos_week'] = np.cos(2 * np.pi * week_series / 52.1429)


In [63]:
# Rolling Window Aggregation (for 4 previous persiods, NaN filled with mean)
table['rolling_4_mean'] = table.groupby(['Store'])['Weekly_Sales'].transform(lambda x: x.shift(1)).rolling(window=4).mean()
# Backfilling the first 4 rows without the mean of previosu 4 weeks
table['rolling_4_mean'] = table.groupby(['Store'])['rolling_4_mean'].bfill()

# Store Data in SQL Database

In [67]:
table.to_sql(name='sales_economic_time_data', con=connection, index=False, if_exists='replace')
connection.close()